<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Setup" data-toc-modified-id="Setup-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Setup</a></span><ul class="toc-item"><li><span><a href="#Check-your-sample-table-(i.e.-metadata-file)" data-toc-modified-id="Check-your-sample-table-(i.e.-metadata-file)-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Check your sample table (i.e. metadata file)</a></span></li><li><span><a href="#Check-your-TRN" data-toc-modified-id="Check-your-TRN-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Check your TRN</a></span></li><li><span><a href="#Load-the-data" data-toc-modified-id="Load-the-data-1.3"><span class="toc-item-num">1.3&nbsp;&nbsp;</span>Load the data</a></span></li></ul></li><li><span><a href="#Regulatory-iModulons" data-toc-modified-id="Regulatory-iModulons-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Regulatory iModulons</a></span></li><li><span><a href="#Functional-iModulons" data-toc-modified-id="Functional-iModulons-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Functional iModulons</a></span><ul class="toc-item"><li><span><a href="#GO-Enrichments" data-toc-modified-id="GO-Enrichments-3.1"><span class="toc-item-num">3.1&nbsp;&nbsp;</span>GO Enrichments</a></span></li><li><span><a href="#KEGG-Enrichments" data-toc-modified-id="KEGG-Enrichments-3.2"><span class="toc-item-num">3.2&nbsp;&nbsp;</span>KEGG Enrichments</a></span><ul class="toc-item"><li><span><a href="#Load-KEGG-mapping" data-toc-modified-id="Load-KEGG-mapping-3.2.1"><span class="toc-item-num">3.2.1&nbsp;&nbsp;</span>Load KEGG mapping</a></span></li><li><span><a href="#Perform-enrichment" data-toc-modified-id="Perform-enrichment-3.2.2"><span class="toc-item-num">3.2.2&nbsp;&nbsp;</span>Perform enrichment</a></span></li><li><span><a href="#Convert-KEGG-IDs-to-human-readable-names" data-toc-modified-id="Convert-KEGG-IDs-to-human-readable-names-3.2.3"><span class="toc-item-num">3.2.3&nbsp;&nbsp;</span>Convert KEGG IDs to human-readable names</a></span></li></ul></li><li><span><a href="#SubtiWiki-categories" data-toc-modified-id="SubtiWiki-categories-3.3"><span class="toc-item-num">3.3&nbsp;&nbsp;</span>SubtiWiki categories</a></span></li><li><span><a href="#Save-files" data-toc-modified-id="Save-files-3.4"><span class="toc-item-num">3.4&nbsp;&nbsp;</span>Save files</a></span></li></ul></li><li><span><a href="#Check-for-single-gene-iModulons" data-toc-modified-id="Check-for-single-gene-iModulons-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Check for single gene iModulons</a></span></li><li><span><a href="#Save-iModulon-object" data-toc-modified-id="Save-iModulon-object-5"><span class="toc-item-num">5&nbsp;&nbsp;</span>Save iModulon object</a></span></li></ul></div>

# Setup
This IPython notebook will walk through the steps of characterizing iModulons through the semi-automated tools in PyModulon. You will need:

* M and A matrices
* Expression data (e.g. `log_tpm_norm.csv`)
* Gene table and KEGG/GO annotations (Generated in `1_create_the_gene_table.ipynb`)
* Sample table, with a column for `project` and `condition`
* TRN file

Optional:
* iModulon table (if you already have some characterized iModulons)

In [1]:
from pymodulon.core import IcaData
from pymodulon.plotting import *
from os import path
import pandas as pd
import re
from Bio.KEGG import REST
from tqdm.notebook import tqdm

In [2]:
# Enter the location of your data here
data_dir = path.join('..', 'ctherm_nf', 'processed_data')

# GO and KEGG annotations are in the 'external' folder
external_data = path.join('..', 'ctherm_nf','external')

In [3]:
import scipy

In [4]:
scipy.__version__

'1.13.1'

## check gene information table

In [3]:
df_gene_table = pd.read_csv(path.join(data_dir,'gene_info.csv'))
df_gene_table

,locus_tag,gene_name,accession,old_locus_tag,start,end,strand,gene_product,COG,uniprot,operon,GO_MF,GO_BP,InterPro,PFAMs
0,Clo1313_0001,dnaA,CP002416.1,NaN,212,1543,+,chromosomal replication initiator protein DnaA,No COG annotation,UPI0001B1F046,Op0,ATP binding; DNA replication origin binding; l...,DNA replication initiation; regulation of DNA ...,AAA+_ATPase;Chromosome_initiator_DnaA;Chromoso...,NaN
1,Clo1313_0002,RS00015,CP002416.1,NaN,1793,2893,+,DNA polymerase III%2C beta subunit,No COG annotation,UPI00017E1F7F,Op1,NaN,NaN,DNA_clamp_sf;DNA_polIII_beta;DNA_polIII_beta_C...,NaN
2,Clo1313_0003,yaaA,CP002416.1,NaN,2909,3115,+,S4 domain protein YaaA,No COG annotation,UPI000038F4E6,Op1,NaN,NaN,RNA-bd_S4-rel_YaaA;S4_RNA-bd;S4_RNA-bd_sf;,NaN
3,Clo1313_0004,recF,CP002416.1,NaN,3133,4242,+,DNA replication and repair protein RecF,No COG annotation,UPI00003C8C54,Op1,ATP binding; single-stranded DNA binding,DNA replication; DNA synthesis involved in DNA...,DNA-binding_RecF;DNA-binding_RecF_CS;P-loop_NT...,NaN
4,Clo1313_0005,RS00030,CP002416.1,NaN,4410,4724,+,hypothetical protein,Function unknown,UPI000038F4E8,Op2,NaN,NaN,RemA-like;,DUF370
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2906,Clo1313_3029,RS15410,CP002416.1,NaN,3558840,3559460,-,single-stranded nucleic acid binding R3H domai...,Function unknown,UPI0001B1F047,Op2051,NaN,NaN,Jag_KH;Jag_N_dom_sf;KH_dom;KH_dom-like_a/b;KH_...,Jag_N;KH_4;R3H
2907,Clo1313_3030,RS15415,CP002416.1,NaN,3559461,3560336,-,membrane protein insertase%2C YidC/Oxa1 family,"Intracellular trafficking, secretion, and vesi...",UPI000038F805,Op2051,NaN,NaN,YidC/ALB3/OXA1/COX18;YidC/Oxa/ALB_C;YidC_ALB_C;,60KD_IMP
2908,Clo1313_3031,yidD,CP002416.1,NaN,3560357,3560572,-,protein of unknown function DUF37,No COG annotation,UPI0000535B71,Op2051,NaN,NaN,Membr_insert_effic_factor_YidD;,NaN
2909,Clo1313_3032,rnpA,CP002416.1,NaN,3560554,3560934,-,ribonuclease P protein component,No COG annotation,UPI00003C8C55,Op2051,3'-tRNA processing endoribonuclease activity; ...,tRNA 5'-leader removal,Ribosomal_Su5_D2-typ_SF;Ribsml_uS5_D2-typ_fold...,NaN


## Check your sample table (i.e. metadata file)
Your metadata file will probably have a lot of columns, most of which you may not care about. Feel free to save a secondary copy of your metadata file with only columns that seem relevant to you. The two most important columns are:
1. `project`
2. `condition`

Make sure that these columns exist in your metadata file

In [4]:
df_metadata = pd.read_csv(path.join(data_dir,'metadata.tsv'),index_col=0,sep='\t')

# for C. thern, I have this information
# df_metadata_p2 = pd.read_csv('../3_quality_control/output_qc/interim/sra_growth_metadata_ctherm_curated_p2.tsv', sep='\t', index_col=0)
# df_metadata = df_metadata_p2.copy()

df_metadata[['project','condition']].head()



,project,condition
SRA_Accession,,
SRX1014220,lacI,del_hpt_del_Clo1313_0089_midlog
SRX1014221,lacI,del_hpt_del_Clo1313_0089_midlog
SRX1014225,lacI,del_hpt_del_Clo1313_0089_sta
SRX1014226,lacI,del_hpt_del_Clo1313_0089_sta
SRX1014227,lacI,del_hpt_del_Clo1313_0089_sta


In [5]:
print(df_metadata.project.notnull().all())
print(df_metadata.condition.notnull().all())

True
True


## Check your TRN
### Not for C. thermocellum, but i have GRN that I will implement. The data was taken from Hebdon et al., Genome-Wide Transcription Factor DNA Binding Sites and Gene Regulatory Networks in Clostridium thermocellum. 2021.

Each row of the TRN file represents a regulatory interaction.  
**Your TRN file must have the following columns:**
1. `regulator` - Name of regulator (`/` or `+` characters will be converted to `;`)
1. `gene_id` - Locus tag of gene being regulated

The following columns are optional, but are helpful to have:
1. `regulator_id` - Locus tag of regulator
1. `gene_name` - Name of gene (can automatically update this using `name2num`)
1. `direction` - Direction of regulation ('+' for activation, '-' for repression, '?' or NaN for unknown)
1. `evidence` - Evidence of regulation (e.g. ChIP-exo, qRT-PCR, SELEX, Motif search)
1. `PMID` - Reference for regulation

You may add any other columns that could help you. TRNs may be saved as either CSV or TSV files. See below for an example:

In [6]:
df_trn = pd.read_csv(path.join(external_data,'GRN_curated.csv'), index_col=0)
print(df_trn.shape)
df_trn.head()

(453, 3)


,regulator,gene_id,effect
0,Xre_0026,Clo1313_0007,NaN
1,Xre_0026,Clo1313_0008,NaN
2,Xre_0026,Clo1313_0046,NaN
3,Xre_0026,Clo1313_0047,NaN
4,Xre_0026,Clo1313_0094,NaN


In [7]:
len(df_trn.regulator.unique())

19

The `regulator` and `gene_id` must be filled in for each row

In [8]:
print(df_trn.regulator.notnull().all())
print(df_trn.gene_id.notnull().all())

True
True


## Load the data
You're now ready to load your IcaData object!

In [12]:
# run this to check the GRN
ica_data = IcaData(M = path.join(data_dir,'M.csv'),
                   A = path.join(data_dir,'A.csv'),
                   X = path.join(data_dir,'log_tpm_norm.csv'),
                   gene_table = path.join(data_dir,'gene_info.csv'),
                   sample_table = path.join(data_dir,'metadata.tsv'),
                   # In C. therm, we do not have TRN, we have GRN from a published study
                   trn = path.join(external_data,'GRN_curated.csv'),
                   optimize_cutoff=True) 

# This gives a more leaner number of imodulons.

If you don't have a TRN (or have a very minimal TRN), use `threshold_method = 'kmeans'`

In [13]:
# # run this first to identify GRN-associated modulons. Then, run it again commenting trn argument for subsequent steps

# ica_data = IcaData(M = path.join(data_dir,'M.csv'),
#                    A = path.join(data_dir,'A.csv'),
#                    X = path.join(data_dir,'log_tpm_norm.csv'),
#                    gene_table = path.join(data_dir,'gene_info.csv'),
#                    sample_table = path.join(data_dir,'metadata.tsv'),
#                    # In C. therm, we do not have TRN, we have GRN from a published study
#                    trn = path.join(external_data,'GRN_curated.csv'),
#                    threshold_method = 'kmeans')

# Regulatory iModulons
### Not for C. thermocellum

Use `compute_trn_enrichment` to automatically check for Regulatory iModulons. The more complete your TRN, the more regulatory iModulons you'll find.

In [14]:
ica_data.compute_trn_enrichment()

,imodulon,regulator,pvalue,qvalue,precision,recall,f1score,TP,regulon_size,imodulon_size,n_regs
0,7,Sigma-16,3.426805e-12,3.426805e-11,0.241379,0.583333,0.341463,7.0,12.0,29.0,1.0
1,42,BlaI_0696,1.105424e-06,5.527119e-06,0.150000,0.750000,0.250000,3.0,4.0,20.0,1.0
2,56,GlyR3,8.521977e-09,1.704395e-08,0.428571,1.000000,0.600000,3.0,3.0,7.0,1.0
3,66,Rex_2471,3.084941e-10,3.084941e-10,0.875000,0.071429,0.132075,7.0,98.0,8.0,1.0


You can also search for AND/OR combinations of regulators using the `max_regs` argument.

Regulator enrichments can be directly saved to the `imodulon_table` using the `save` argument. This saves the enrichment with the lowest q-value to the table.

In [15]:
# First search for regulator enrichments with 2 regulators
ica_data.compute_trn_enrichment(max_regs=2,save=True)

# Next, search for regulator enrichments with just one regulator. This will supercede the 2 regulator enrichments.
ica_data.compute_trn_enrichment(max_regs=1,save=True)

,imodulon,regulator,pvalue,qvalue,precision,recall,f1score,TP,regulon_size,imodulon_size,n_regs
0,7,Sigma-16,3.426805e-12,3.426805e-11,0.241379,0.583333,0.341463,7.0,12.0,29.0,1.0
1,42,BlaI_0696,1.105424e-06,5.527119e-06,0.150000,0.750000,0.250000,3.0,4.0,20.0,1.0
2,56,GlyR3,8.521977e-09,1.704395e-08,0.428571,1.000000,0.600000,3.0,3.0,7.0,1.0
3,66,Rex_2471,3.084941e-10,3.084941e-10,0.875000,0.071429,0.132075,7.0,98.0,8.0,1.0


The list of regulatory iModulons are shown below

In [16]:
regulatory_imodulons = ica_data.imodulon_table[ica_data.imodulon_table.regulator.notnull()]
print(len(ica_data.imodulon_table),'Total iModulons')
print(len(regulatory_imodulons),'Regulatory iModulons')

regulatory_imodulons

67 Total iModulons
4 Regulatory iModulons


,regulator,pvalue,qvalue,precision,recall,f1score,TP,regulon_size,imodulon_size,n_regs
7,Sigma-16,3.426805e-12,3.426805e-11,0.241379,0.583333,0.341463,7.0,12.0,29.0,1.0
42,BlaI_0696,1.105424e-06,5.527119e-06,0.150000,0.750000,0.250000,3.0,4.0,20.0,1.0
56,GlyR3,8.521977e-09,1.704395e-08,0.428571,1.000000,0.600000,3.0,3.0,7.0,1.0
66,Rex_2471,3.084941e-10,3.084941e-10,0.875000,0.071429,0.132075,7.0,98.0,8.0,1.0


In [17]:
regulatory_imodulons_to_save = regulatory_imodulons.copy()
regulatory_imodulons_to_save.reset_index().to_csv( '../ctherm_nf/interim/grn_from_ica.tsv', sep='\t')

You can rename iModulons in this jupyter notebook, or you can save the iModulon table as a CSV and edit it in Excel.

If two iModulons have the same regulator (e.g. 'Reg'), they will be named 'Reg-1' and 'Reg-2'

In [18]:
# ica_data.rename_imodulons(regulatory_imodulons.regulator.to_dict())
# ica_data.imodulon_table.head()

In [19]:
# regulatory_imodulons = ica_data.imodulon_table[ica_data.imodulon_table.regulator.notnull()]

### find large imodulons and manually change thresholds

In [21]:
# # Get size of each iModulon
# imodulon_sizes = {im: len(ica_data.view_imodulon(im)) 
#                   for im in ica_data.imodulon_names}

# # Sort by size descending
# size_df = pd.Series(imodulon_sizes).sort_values(ascending=False)
# print(size_df.head(20))  # Top 20 largest

# Functional iModulons

GO annotations and KEGG pathways/modules were generated in the 1_create_the_gene_table.ipynb notebook. Enrichments will be calculated in this notebook, and further curated in the 3_manual_iModulon_curation notebook.

## GO Enrichments

First load the Gene Ontology annotations

In [22]:
# Molecular Function
DF_GO_mf = pd.read_csv(path.join(external_data,'GO_annotations_MF_curated.csv'),index_col=0)
DF_GO_mf_enrich = ica_data.compute_annotation_enrichment(DF_GO_mf,'gene_ontology')

# Biological Process
DF_GO_bp = pd.read_csv(path.join(external_data,'GO_annotations_BP_curated.csv'),index_col=0)
DF_GO_bp_enrich = ica_data.compute_annotation_enrichment(DF_GO_bp,'gene_ontology')

### Biocyc pathways

In [23]:
DF_biocyc = pd.read_csv(path.join(external_data,'biocyc_pathways_curated.csv'),index_col=0)

DF_biocyc = DF_biocyc.loc[DF_biocyc['gene_id'].isin(ica_data.gene_names)]

DF_biocyc.head()

,gene_id,biocyc_pathways
0,Clo1313_2606,tRNA charging
1,Clo1313_2723,tRNA charging
2,Clo1313_1301,tRNA charging
3,Clo1313_1578,tRNA charging
4,Clo1313_1434,tRNA charging


In [24]:
DF_biocyc_enrich = ica_data.compute_annotation_enrichment(DF_biocyc,'biocyc_pathways')

In [25]:
DF_biocyc_enrich.head()

,imodulon,biocyc_pathways,pvalue,qvalue,precision,recall,f1score,TP,target_set_size,imodulon_size
0,2,L-tryptophan biosynthesis,2.510499e-11,4.744843e-09,0.800000,0.666667,0.727273,4.0,6.0,5.0
1,7,"(1,4)-&beta;-D-xylan degradation",9.408885e-04,8.891397e-02,0.068966,0.400000,0.117647,2.0,5.0,29.0
2,7,cellulose and hemicellulose degradation (cellu...,9.408885e-04,8.891397e-02,0.068966,0.400000,0.117647,2.0,5.0,29.0
3,10,fatty acid biosynthesis initiation (type II),6.927798e-07,1.309354e-04,0.230769,0.600000,0.333333,3.0,5.0,13.0
4,10,even <i>iso</i>-branched-chain fatty acid bios...,3.849618e-06,3.637889e-04,0.230769,0.375000,0.285714,3.0,8.0,13.0


## KEGG Enrichments

### Load KEGG mapping
The `kegg_mapping.csv` file contains KEGG orthologies, pathways, modules, and reactions. Only pathways and modules are relevant to iModulon characterization.

In [26]:
DF_KEGG = pd.read_csv(path.join(external_data,'kegg_mapping.csv'),index_col=0)
print(DF_KEGG.database.unique())
DF_KEGG.head()

['KEGG_pathway' 'KEGG_module' 'KEGG_reaction']


,gene_id,database,kegg_id
0,Clo1313_0005,KEGG_pathway,-
1,Clo1313_0006,KEGG_pathway,-
2,Clo1313_0007,KEGG_pathway,-
3,Clo1313_0008,KEGG_pathway,-
4,Clo1313_0009,KEGG_pathway,-


In [27]:
kegg_pathways = DF_KEGG[DF_KEGG.database == 'KEGG_pathway']
kegg_modules = DF_KEGG[DF_KEGG.database == 'KEGG_module']

In [28]:
kegg_pathways = kegg_pathways.loc[kegg_pathways.kegg_id!='-']
kegg_pathways

,gene_id,database,kegg_id
7,Clo1313_0011,KEGG_pathway,map00970
14,Clo1313_0016,KEGG_pathway,map01502
15,Clo1313_0016,KEGG_pathway,map02020
31,Clo1313_0020,KEGG_pathway,map00010
32,Clo1313_0020,KEGG_pathway,map00020
...,...,...,...
7590,Clo1313_3023,KEGG_pathway,map00500
7591,Clo1313_3023,KEGG_pathway,map01100
7601,Clo1313_3030,KEGG_pathway,map02024
7602,Clo1313_3030,KEGG_pathway,map03060


In [29]:
kegg_modules = kegg_modules.loc[kegg_modules.kegg_id!='-']
kegg_modules

,gene_id,database,kegg_id
7610,Clo1313_0011,KEGG_module,M00359
7611,Clo1313_0011,KEGG_module,M00360
7616,Clo1313_0016,KEGG_module,M00651
7620,Clo1313_0020,KEGG_module,M00173
7621,Clo1313_0020,KEGG_module,M00307
...,...,...,...
10666,Clo1313_2994,KEGG_module,M00459
10688,Clo1313_3017,KEGG_module,M00115
10689,Clo1313_3018,KEGG_module,M00115
10690,Clo1313_3019,KEGG_module,M00115


### Perform enrichment
Uses the `compute_annotation_enrichment` function

In [30]:
DF_pathway_enrich = ica_data.compute_annotation_enrichment(kegg_pathways,'kegg_id')
DF_module_enrich = ica_data.compute_annotation_enrichment(kegg_modules,'kegg_id')

In [31]:
DF_pathway_enrich.head()

,imodulon,kegg_id,pvalue,qvalue,precision,recall,f1score,TP,target_set_size,imodulon_size
0,2,map00400,4.650481e-11,8.928923e-09,1.0,0.185185,0.312500,5.0,27.0,5.0
1,2,map01230,7.382974e-08,7.087655e-06,1.0,0.045045,0.086207,5.0,111.0,5.0
2,2,map01130,4.157131e-07,2.660564e-05,1.0,0.032051,0.062112,5.0,156.0,5.0
3,2,map01110,1.188473e-06,5.704673e-05,1.0,0.026042,0.050761,5.0,192.0,5.0
4,2,map00405,2.360994e-06,9.066215e-05,0.4,1.000000,0.571429,2.0,2.0,5.0


In [32]:
DF_module_enrich.head()

,imodulon,kegg_id,pvalue,qvalue,precision,recall,f1score,TP,target_set_size,imodulon_size
0,2,M00023,3.225900e-14,6.355023e-12,1.000000,0.625000,0.769231,5.0,8.0,5.0
1,10,M00083,3.849618e-06,7.583747e-04,0.230769,0.375000,0.285714,3.0,8.0,13.0
2,10,M00082,1.099381e-04,1.082891e-02,0.153846,0.500000,0.235294,2.0,4.0,13.0
3,10,M00572,3.818813e-04,2.507687e-02,0.153846,0.285714,0.200000,2.0,7.0,13.0
4,12,M00212,2.716494e-07,5.351494e-05,0.375000,0.500000,0.428571,3.0,6.0,8.0


### Convert KEGG IDs to human-readable names

In [33]:
for idx,key in tqdm(DF_pathway_enrich.kegg_id.items(),total=len(DF_pathway_enrich)):
    if '-' not in key:
        text = REST.kegg_find('pathway',key).read()
    try:
        name = re.search('\t(.*)\n',text).group(1)
        DF_pathway_enrich.loc[idx,'pathway_name'] = name
    except AttributeError:
        DF_pathway_enrich.loc[idx,'pathway_name'] = None
    
for idx,key in tqdm(DF_module_enrich.kegg_id.items(),total=len(DF_module_enrich)):
    if '-' not in key:
        text = REST.kegg_find('module',key).read()
    try:
        name = re.search('\t(.*)\n',text).group(1)
        DF_module_enrich.loc[idx,'module_name'] = name
    except AttributeError:
        DF_module_enrich.loc[idx,'module_name'] = None

  0%|          | 0/166 [00:00<?, ?it/s]

  0%|          | 0/78 [00:00<?, ?it/s]

In [45]:
DF_pathway_enrich.head()

,imodulon,kegg_id,pvalue,qvalue,precision,recall,f1score,TP,target_set_size,imodulon_size,pathway_name
0,2,map00400,4.650481e-11,8.928923e-09,1.0,0.185185,0.312500,5.0,27.0,5.0,"Phenylalanine, tyrosine and tryptophan biosynt..."
1,2,map01230,7.382974e-08,7.087655e-06,1.0,0.045045,0.086207,5.0,111.0,5.0,Biosynthesis of amino acids
2,2,map01130,4.157131e-07,2.660564e-05,1.0,0.032051,0.062112,5.0,156.0,5.0,None
3,2,map01110,1.188473e-06,5.704673e-05,1.0,0.026042,0.050761,5.0,192.0,5.0,Biosynthesis of secondary metabolites
4,2,map00405,2.360994e-06,9.066215e-05,0.4,1.000000,0.571429,2.0,2.0,5.0,Phenazine biosynthesis


In [46]:
DF_module_enrich.head()

,imodulon,kegg_id,pvalue,qvalue,precision,recall,f1score,TP,target_set_size,imodulon_size,module_name
0,2,M00023,3.225900e-14,6.355023e-12,1.000000,0.625000,0.769231,5.0,8.0,5.0,"Tryptophan biosynthesis, chorismate => tryptophan"
1,10,M00083,3.849618e-06,7.583747e-04,0.230769,0.375000,0.285714,3.0,8.0,13.0,"Fatty acid biosynthesis, elongation"
2,10,M00082,1.099381e-04,1.082891e-02,0.153846,0.500000,0.235294,2.0,4.0,13.0,"Fatty acid biosynthesis, initiation"
3,10,M00572,3.818813e-04,2.507687e-02,0.153846,0.285714,0.200000,2.0,7.0,13.0,"Pimeloyl-ACP biosynthesis, BioC-BioH pathway, ..."
4,12,M00212,2.716494e-07,5.351494e-05,0.375000,0.500000,0.428571,3.0,6.0,8.0,None


## SubtiWiki categories - Not for C. therm

In [51]:
# DF_subtiwiki = pd.read_csv(path.join(external_data,'subtiwiki_categories.csv'))
# DF_subtiwiki.head()

In [52]:
# # Change the subtiwiki annotation format into a list of genes and categories
# DF_subtiwiki = DF_subtiwiki.rename({'BSU_number':'gene_id'},axis=1)
# DF_subtiwiki = DF_subtiwiki.melt(id_vars='gene_id',value_vars=['FuncName1','FuncName2','FuncName3','FuncName4','FuncName5'])
# DF_subtiwiki = DF_subtiwiki[DF_subtiwiki.value.notnull() & DF_subtiwiki.gene_id.isin(ica_data.gene_names)]
# DF_subtiwiki.head()

In [53]:
# DF_subti_enrich = ica_data.compute_annotation_enrichment(DF_subtiwiki,'value')

## Interpro and PFAM

In [54]:
DF_interpro = pd.read_csv(path.join(external_data,'interpro_curated.csv'),index_col=0)
DF_interpro_enrich = ica_data.compute_annotation_enrichment(DF_interpro,'interpro_names')

DF_interpro_enrich.head()

,imodulon,interpro_names,pvalue,qvalue,precision,recall,f1score,TP,target_set_size,imodulon_size
0,0,Tnp_DDE_dom,6.749406e-06,3.155347e-02,0.053571,1.000000,0.101695,3.0,3.0,56.0
1,7,Dockerin_1_rpt,3.793325e-31,5.911265e-28,0.758621,0.297297,0.427184,22.0,74.0,29.0
2,7,Dockerin_dom,2.671877e-31,5.911265e-28,0.758621,0.301370,0.431373,22.0,73.0,29.0
3,7,Dockerin_dom_sf,3.793325e-31,5.911265e-28,0.758621,0.297297,0.427184,22.0,74.0,29.0
4,7,EF_Hand_1_Ca_BS,5.083463e-21,5.941297e-18,0.551724,0.271186,0.363636,16.0,59.0,29.0


In [55]:
DF_pfam = pd.read_csv(path.join(external_data,'pfam_curated.csv'),index_col=0)
DF_pfam_enrich = ica_data.compute_annotation_enrichment(DF_pfam,'PFAMs')

DF_pfam_enrich.head()

,imodulon,PFAMs,pvalue,qvalue,precision,recall,f1score,TP,target_set_size,imodulon_size
0,0,DDE_Tnp_1_3,2.200419e-09,4.209401e-06,0.089286,1.000000,0.163934,5.0,5.0,56.0
1,7,Dockerin_1,7.034337e-27,1.345669e-23,0.655172,0.316667,0.426966,19.0,60.0,29.0
2,7,CBM_6,5.162249e-08,4.937691e-05,0.172414,0.416667,0.243902,5.0,12.0,29.0
3,7,AbfB,3.534914e-06,1.127048e-03,0.103448,0.750000,0.181818,3.0,4.0,29.0
4,7,CBM_35,3.534914e-06,1.127048e-03,0.103448,0.750000,0.181818,3.0,4.0,29.0


## DAP-inferred GRN

In [2]:
DF_GRN = pd.read_csv(path.join(external_data,'GRN_curated.csv'),index_col=0)

# ensure the genes are present in the ica_data
DF_GRN = DF_GRN.loc[DF_GRN['gene_id'].isin(ica_data.gene_names)]

DF_GRN.head()

NameError: name 'pd' is not defined

In [57]:
len(DF_GRN.regulator.unique())

18

In [58]:
DF_grn_enrich = ica_data.compute_annotation_enrichment(DF_GRN,'regulator')

In [59]:
DF_grn_enrich

,imodulon,regulator,pvalue,qvalue,precision,recall,f1score,TP,target_set_size,imodulon_size
0,7,Sigma-16,3.426805e-12,6.168249e-11,0.241379,0.583333,0.341463,7.0,12.0,29.0
1,7,Sigma-13,3.534914e-06,3.181422e-05,0.103448,0.750000,0.181818,3.0,4.0,29.0
2,7,Sigma-17,9.585634e-05,5.751380e-04,0.068966,1.000000,0.129032,2.0,2.0,29.0
3,25,LexA_1449,1.959852e-05,3.527733e-04,0.200000,0.094595,0.128440,7.0,74.0,35.0
4,25,Xre_0026,1.418118e-03,1.276306e-02,0.114286,0.095238,0.103896,4.0,42.0,35.0
5,25,Fur_1691,1.600079e-02,9.600477e-02,0.085714,0.066667,0.075000,3.0,45.0,35.0
6,39,Xre_2225,1.936015e-04,3.484826e-03,0.048780,1.000000,0.093023,2.0,2.0,41.0
7,42,BlaI_0696,1.105424e-06,1.989763e-05,0.150000,0.750000,0.250000,3.0,4.0,20.0
8,46,TetR_0692,2.019076e-04,3.634337e-03,0.056338,0.228571,0.090395,8.0,35.0,142.0
9,48,GlyR3,5.510798e-05,9.919436e-04,0.153846,0.666667,0.250000,2.0,3.0,13.0


## Save files

In [60]:
# SOURCES
DF_biocyc_enrich['source'] = 'biocyc'
DF_pathway_enrich['source'] = 'kegg_pathway'
DF_module_enrich['source'] = 'kegg_module'
DF_grn_enrich['source'] = 'GRN'
# INTERPRO AND PFAMs
DF_interpro_enrich['source'] = 'interpro'
DF_pfam_enrich['source'] = 'pfam'
# GO
DF_GO_mf_enrich['source'] = 'GO_MF'
DF_GO_bp_enrich['source'] = 'GO_BP'


# Make the columns annotations
DF_biocyc_enrich.rename({'biocyc_pathways':'annotation'},axis=1, inplace=True)
DF_pathway_enrich.rename({'kegg_id':'annotation'},axis=1, inplace=True)
DF_module_enrich.rename({'kegg_id':'annotation'},axis=1, inplace=True)
DF_grn_enrich.rename({'regulator':'annotation'},axis=1, inplace=True)
DF_GO_mf_enrich.rename({'gene_ontology':'annotation'},axis=1, inplace=True)
DF_GO_bp_enrich.rename({'gene_ontology':'annotation'},axis=1, inplace=True)
DF_interpro_enrich.rename({'interpro_names':'annotation'},axis=1, inplace=True)
DF_pfam_enrich.rename({'PFAMs':'annotation'},axis=1, inplace=True)

# concat
DF_enrichments = pd.concat([DF_biocyc_enrich, DF_pathway_enrich, DF_module_enrich, DF_GO_mf_enrich, DF_GO_bp_enrich, DF_interpro_enrich, DF_pfam_enrich, DF_grn_enrich])


In [61]:
DF_enrichments

,imodulon,annotation,pvalue,qvalue,precision,recall,f1score,TP,target_set_size,imodulon_size,source,pathway_name,module_name
0,2,L-tryptophan biosynthesis,2.510499e-11,4.744843e-09,0.800000,0.666667,0.727273,4.0,6.0,5.0,biocyc,NaN,NaN
1,7,"(1,4)-&beta;-D-xylan degradation",9.408885e-04,8.891397e-02,0.068966,0.400000,0.117647,2.0,5.0,29.0,biocyc,NaN,NaN
2,7,cellulose and hemicellulose degradation (cellu...,9.408885e-04,8.891397e-02,0.068966,0.400000,0.117647,2.0,5.0,29.0,biocyc,NaN,NaN
3,10,fatty acid biosynthesis initiation (type II),6.927798e-07,1.309354e-04,0.230769,0.600000,0.333333,3.0,5.0,13.0,biocyc,NaN,NaN
4,10,even <i>iso</i>-branched-chain fatty acid bios...,3.849618e-06,3.637889e-04,0.230769,0.375000,0.285714,3.0,8.0,13.0,biocyc,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9,48,GlyR3,5.510798e-05,9.919436e-04,0.153846,0.666667,0.250000,2.0,3.0,13.0,GRN,NaN,NaN
10,49,GlyR2,5.177310e-03,9.319158e-02,0.100000,0.085714,0.092308,3.0,35.0,30.0,GRN,NaN,NaN
11,54,LexA_1449,1.369519e-03,2.465135e-02,0.200000,0.054054,0.085106,4.0,74.0,20.0,GRN,NaN,NaN
12,56,GlyR3,8.521977e-09,1.533956e-07,0.428571,1.000000,0.600000,3.0,3.0,7.0,GRN,NaN,NaN


In [62]:
# save
DF_enrichments.to_csv(path.join(data_dir,'functional_enrichments.csv'))

# Check for single gene iModulons

### Do not rename the values because it causes issues later

Some iModulons are dominated by a single, high-coefficient gene. These iModulons may result from:
1. Overdecomposition of the dataset to identify noisy genes
1. Artificial knock-out of single genes
1. Regulons with only one target gene

No matter what causes these iModulons, it is important to be aware of them. The find_single_gene_imodulons function identifies iModulons that are likely dominated by a single gene.

The iModulons identified by ``find_single_gene_imodulons`` may contain more than one gene, since a threshold-agnostic method is used to identify these iModulons.

In [63]:
sg_imods = ica_data.find_single_gene_imodulons(save=True)
len(sg_imods)

5

In [64]:
sg_imods

[4, 17, 31, 33, 56]

In [65]:
# # # Do not rename the values because it causes issues later
# for i,mod in enumerate(sg_imods):
#     ica_data.rename_imodulons({mod:'SG_'+str(i+1)})

In [66]:
sg_data = ica_data.imodulon_table[ica_data.imodulon_table.single_gene == True]

In [67]:
for x in sg_data.index.to_list():
    print(x, ica_data.view_imodulon(x).index.to_list())

4 ['Clo1313_0124', 'Clo1313_0401', 'Clo1313_0402', 'Clo1313_0483', 'Clo1313_0484', 'Clo1313_1708', 'Clo1313_1709', 'Clo1313_2323', 'Clo1313_2842']
17 ['Clo1313_1160', 'Clo1313_2927']
31 ['Clo1313_1160', 'Clo1313_1857']
33 ['Clo1313_0576', 'Clo1313_0606', 'Clo1313_2034', 'Clo1313_2035', 'Clo1313_2038', 'Clo1313_2260', 'Clo1313_2303', 'Clo1313_2484', 'Clo1313_2485', 'Clo1313_2486', 'Clo1313_2488']
56 ['Clo1313_0089', 'Clo1313_0205', 'Clo1313_0395', 'Clo1313_0396', 'Clo1313_0397', 'Clo1313_1398', 'Clo1313_1485']


# Save iModulon object

In [68]:
from pymodulon.util import explained_variance
from pymodulon.io import *

In [69]:
# Add iModulon sizes and explained variance
for im in ica_data.imodulon_names:
    ica_data.imodulon_table.loc[im,'imodulon_size'] = len(ica_data.view_imodulon(im))
    ica_data.imodulon_table.loc[im,'explained_variance'] = explained_variance(ica_data,imodulons=im)

In [70]:
# ## NO RENAMING NEEDED
# # ica_data.imodulon_table.reset_index(inplace=True, drop=False)
# ica_data.imodulon_table.rename(columns={'imodulons':'imodulon_id'}, inplace=True)

In [71]:
total_variance = np.round(ica_data.imodulon_table['explained_variance'].sum(), 2)

print(f'total variance explained by all iModulons: {total_variance}')

print(f'Total imodulons: {len(ica_data.imodulon_names)}')

ica_data.imodulon_table.sort_values('explained_variance', ascending=False)

total variance explained by all iModulons: 0.8
Total imodulons: 67


,regulator,pvalue,qvalue,precision,recall,f1score,TP,regulon_size,imodulon_size,n_regs,single_gene,explained_variance
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71.0,NaN,NaN,0.059580
45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.0,NaN,NaN,0.039654
49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,NaN,NaN,0.038669
28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72.0,NaN,NaN,0.028296
27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.0,NaN,NaN,0.028090
...,...,...,...,...,...,...,...,...,...,...,...,...
14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,0.003941
56,GlyR3,8.521977e-09,1.704395e-08,0.428571,1.0,0.6,3.0,3.0,7.0,1.0,True,0.002189
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.0,NaN,NaN,0.001555
53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0,NaN,NaN,0.001193


In [72]:
ica_data.imodulon_table.loc[ica_data.imodulon_table.single_gene==True]

,regulator,pvalue,qvalue,precision,recall,f1score,TP,regulon_size,imodulon_size,n_regs,single_gene,explained_variance
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.0,NaN,True,0.006902
17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,True,0.018437
31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,True,0.004669
33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,NaN,True,0.006838
56,GlyR3,8.521977e-09,1.704395e-08,0.428571,1.0,0.6,3.0,3.0,7.0,1.0,True,0.002189


This will save your iModulon table, your thresholds, and any other information stored in the ica_data object.

In [73]:
df_grn_ica = pd.read_csv('../ctherm_nf/interim/grn_from_ica.tsv', sep='\t', index_col=0)
df_grn_ica
df_grn_ica.set_index('index', inplace=True)
dict_grn_ica = df_grn_ica[['regulator']].to_dict()['regulator']
dict_grn_ica_rev = {v:str(k) for k,v in dict_grn_ica.items()}
dict_grn_ica_rev

{'Sigma-16': '7', 'BlaI_0696': '42', 'GlyR3': '56', 'Rex_2471': '66'}

In [74]:
# # rename regulators, otherwise won't load
# ica_data.rename_imodulons(dict_grn_ica_rev)

In [75]:
save_to_json(ica_data, path.join('..', 'ctherm_nf', 'interim','ctherm_raw.json.gz'))

If you prefer to view and edit your iModulon table in excel, save it as a CSV and reload the iModulon as before

In [62]:
ica_data.imodulon_table.to_csv(path.join('..', 'ctherm_nf','interim','imodulon_table_raw.csv'))

In [ ]:
# Gene: Clo1313_0963 could be "5‑formyltetrahydrofolate cyclo‑ligase"
# Gene: Clo1313_1220 not sure "hypothetical protein"

In [ ]:
ica_data.imodulon_table.loc[15]

In [ ]:
ica_data.imodulon_table.sort_values('explained_variance', ascending=False)

### FIN